
# ARC-AGI DSL Agent Notebook

This notebook mirrors the structure of the in-class TALE-Suite notebook, but replaces the text-adventure environment with an ARC-AGI **step-by-step DSL environment**.

## What this notebook does

For each ARC puzzle:

1. Loads the training input/output grid pairs.
2. Loads the target test input grid.
3. Extracts the **ground-truth solver trajectory** from `solvers.py`.
4. Restricts the action space to the **DSL steps that appear in that puzzle's solver**.
5. Lets an LLM agent choose **one step at a time**.
6. Executes the chosen step on the current symbolic state.
7. Stops when the produced `O` exactly matches the hidden test output, or when the step budget is exhausted.

## Important design choice

Some ARC solver trajectories are not just single unary grid transforms.  
They often include intermediate variables such as:

```python
x1 = objects(I, T, F, F)
x2 = colorfilter(x1, ZERO)
...
O = fill(I, FOUR, x5)
```

So in this notebook, each action is an **executable solver step** (one assignment line), not just a raw function name. This makes the ARC environment faithful to the provided `solve_{puzzle_id}` trajectory while still keeping the action space very small.



## Workspace assumptions

This notebook expects the following files in your workspace:

- `dsl.py`
- `solvers.py`
- `puzzle_ids.py`
- `data/{puzzle_id}.json`

It also works with the attached sample puzzle `00d62c1b.json`.

The API section mirrors your in-class Triton notebook:
- endpoint: `https://tritonai-api.ucsd.edu/v1/chat/completions`
- model: configurable
- key: from Colab userdata or environment variable


In [2]:

from __future__ import annotations

import ast
import json
import os
import pathlib
import re
import sys
import time
import types
import typing
from dataclasses import dataclass, field

import requests


In [1]:

# --- Configuration ---
WORKSPACE_DIR = pathlib.Path(".")
DATA_DIR = WORKSPACE_DIR / "data"
DSL_PATH = WORKSPACE_DIR / "dsl.py"
SOLVERS_PATH = WORKSPACE_DIR / "solvers.py"
PUZZLE_IDS_PATH = WORKSPACE_DIR / "puzzle_ids.py"

API_URL = "https://tritonai-api.ucsd.edu/v1/chat/completions"
MODEL = "api-gpt-oss-120b"
MAX_STEPS = 30
SEED = 20241001
REASONING_EFFORT = 1024
TEMPERATURE = 0.0

# try:
#     from google.colab import userdata  # type: ignore
#     API_KEY = userdata.get("TritonAPI")
# except Exception:
#     API_KEY = os.getenv("TRITON_API_KEY", "")
API_KEY = os.environ.get("OPENAI_API_KEY")
print(API_KEY)

NameError: name 'pathlib' is not defined

## Helpers for loading ARC DSL, puzzle metadata, and solver steps

In [5]:

def install_dummy_arc_types() -> None:
    """Creates a lightweight fallback `arc_types` module if the real one is missing.

    `dsl.py` imports `from arc_types import *`. In many ARC repos these names are
    only used for type hints, so a minimal runtime stub is enough.
    """
    if "arc_types" in sys.modules:
        return

    m = types.ModuleType("arc_types")
    fallback_names = {
        "Any": typing.Any,
        "Callable": typing.Callable,
        "FrozenSet": typing.FrozenSet,
        "Tuple": typing.Tuple,
        "Container": typing.Container,
        "Boolean": bool,
        "Integer": int,
        "IntegerTuple": tuple,
        "Numerical": object,
        "IntegerSet": object,
        "ContainerContainer": object,
        "TupleTuple": tuple,
        "Element": object,
        "Piece": object,
        "Patch": object,
        "Object": object,
        "Objects": object,
        "Indices": object,
        "Index": tuple,
        "Grid": tuple,
        "Cell": tuple,
    }
    for name, value in fallback_names.items():
        setattr(m, name, value)
    sys.modules["arc_types"] = m


def load_dsl_namespace(dsl_path: pathlib.Path) -> dict:
    install_dummy_arc_types()
    namespace: dict = {}
    exec(dsl_path.read_text(encoding="utf-8"), namespace)
    return namespace


def fallback_constants() -> dict:
    return {
        "F": False,
        "T": True,
        "ZERO": 0,
        "ONE": 1,
        "TWO": 2,
        "THREE": 3,
        "FOUR": 4,
        "FIVE": 5,
        "SIX": 6,
        "SEVEN": 7,
        "EIGHT": 8,
        "NINE": 9,
        "TEN": 10,
        "NEG_ONE": -1,
        "ORIGIN": (0, 0),
        "UNITY": (1, 1),
        "UP": (-1, 0),
        "DOWN": (1, 0),
        "LEFT": (0, -1),
        "RIGHT": (0, 1),
        "UP_LEFT": (-1, -1),
        "UP_RIGHT": (-1, 1),
        "DOWN_LEFT": (1, -1),
        "DOWN_RIGHT": (1, 1),
        "ZERO_BY_TWO": (0, 2),
        "TWO_BY_ZERO": (2, 0),
        "TWO_BY_TWO": (2, 2),
        "THREE_BY_THREE": (3, 3),
        "FOUR_BY_FOUR": (4, 4),
    }


def load_constants_namespace(workspace_dir: pathlib.Path) -> dict:
    constants_path = workspace_dir / "constants.py"
    if constants_path.exists():
        namespace: dict = {}
        exec(constants_path.read_text(encoding="utf-8"), namespace)
        return namespace
    return fallback_constants()


def load_puzzle_ids(puzzle_ids_path: pathlib.Path) -> dict:
    namespace: dict = {}
    exec(puzzle_ids_path.read_text(encoding="utf-8"), namespace)
    return {
        "TRAIN_IDS": namespace.get("TRAIN_IDS", []),
        "TEST_IDS": namespace.get("TEST_IDS", []),
        "ALL_IDS": namespace.get("ALL_IDS", []),
    }


def to_grid(grid_like):
    return tuple(tuple(int(v) for v in row) for row in grid_like)



def resolve_puzzle_path(data_dir: pathlib.Path, puzzle_id: str) -> pathlib.Path:
    candidate_paths = [
        data_dir / f"{puzzle_id}.json",
        WORKSPACE_DIR / f"{puzzle_id}.json",
    ]
    for path in candidate_paths:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {puzzle_id}.json in {data_dir} or {WORKSPACE_DIR}"
    )


def load_puzzle_json(data_dir: pathlib.Path, puzzle_id: str) -> dict:
    path = resolve_puzzle_path(data_dir, puzzle_id)
    raw = json.loads(path.read_text(encoding="utf-8"))
    return {
        "train": [
            {
                "input": to_grid(example["input"]),
                "output": to_grid(example["output"]),
            }
            for example in raw["train"]
        ],
        "test": [
            {
                "input": to_grid(example["input"]),
                "output": to_grid(example["output"]),
            }
            for example in raw["test"]
        ],
    }


def extract_solver_steps(solvers_path: pathlib.Path, puzzle_id: str) -> list[dict]:
    source = solvers_path.read_text(encoding="utf-8")
    tree = ast.parse(source)
    function_name = f"solve_{puzzle_id}"

    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            steps = []
            for stmt in node.body:
                if not isinstance(stmt, ast.Assign):
                    continue

                step_source = ast.get_source_segment(source, stmt).strip()
                target = stmt.targets[0].id if isinstance(stmt.targets[0], ast.Name) else None
                dependencies = sorted(
                    {
                        n.id
                        for n in ast.walk(stmt.value)
                        if isinstance(n, ast.Name)
                    }
                )
                call_names = []
                for n in ast.walk(stmt.value):
                    if isinstance(n, ast.Call) and isinstance(n.func, ast.Name):
                        call_names.append(n.func.id)

                steps.append(
                    {
                        "id": len(steps) + 1,
                        "source": step_source,
                        "target": target,
                        "dependencies": dependencies,
                        "call_names": call_names,
                    }
                )
            return steps

    raise KeyError(f"Could not find solve_{puzzle_id} in {solvers_path}")


def ordered_unique(items):
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


def get_helper_dsl_docs(steps: list[dict], dsl_namespace: dict) -> list[tuple[str, str]]:
    names = ordered_unique(
        name
        for step in steps
        for name in step["call_names"]
    )

    docs = []
    for name in names:
        obj = dsl_namespace.get(name)
        doc = ""
        if callable(obj) and getattr(obj, "__doc__", None):
            doc = obj.__doc__.strip().replace("\n", " ")
        docs.append((name, doc))
    return docs


def grid_to_text(grid) -> str:
    return "\n".join(" ".join(str(v) for v in row) for row in grid)


def is_grid(value) -> bool:
    return (
        isinstance(value, tuple)
        and (len(value) == 0 or all(isinstance(row, tuple) for row in value))
    )


def compare_grids(candidate, target) -> tuple[int, int]:
    if candidate is None or target is None:
        return 0, 0
    total = len(target) * len(target[0])
    matched = sum(
        int(candidate[i][j] == target[i][j])
        for i in range(len(target))
        for j in range(len(target[0]))
    )
    return matched, total


def parse_option_id(text: str) -> int | None:
    match = re.search(r"\d+", str(text))
    return int(match.group()) if match else None


def summarize_value(value, max_chars: int = 240) -> str:
    if is_grid(value):
        h = len(value)
        w = len(value[0]) if h else 0
        return f"grid[{h}x{w}]\n{grid_to_text(value)}"

    if isinstance(value, (set, frozenset, tuple, list)):
        preview = list(value)[:3]
        text = f"{type(value).__name__}(len={len(value)}): {preview!r}"
        return text[:max_chars]

    return repr(value)[:max_chars]


PUZZLE_SPLITS = load_puzzle_ids(PUZZLE_IDS_PATH)
DSL_NAMESPACE = load_dsl_namespace(DSL_PATH)
CONSTANTS_NAMESPACE = load_constants_namespace(WORKSPACE_DIR)


## ARC DSL environment

In [6]:

@dataclass
class ARCStepEnv:
    puzzle_id: str
    data_dir: pathlib.Path
    solvers_path: pathlib.Path
    dsl_namespace: dict
    constants_namespace: dict
    max_steps: int = 30

    puzzle: dict = field(init=False)
    steps: list[dict] = field(init=False)
    helper_docs: list[tuple[str, str]] = field(init=False)
    globals_for_exec: dict = field(init=False)

    remaining_step_ids: set[int] = field(init=False, default_factory=set)
    vars: dict = field(init=False, default_factory=dict)
    history: list[dict] = field(init=False, default_factory=list)
    done: bool = field(init=False, default=False)
    step_count: int = field(init=False, default=0)

    def __post_init__(self):
        self.puzzle = load_puzzle_json(self.data_dir, self.puzzle_id)
        self.steps = extract_solver_steps(self.solvers_path, self.puzzle_id)
        self.helper_docs = get_helper_dsl_docs(self.steps, self.dsl_namespace)
        self.globals_for_exec = {}
        self.globals_for_exec.update(self.dsl_namespace)
        self.globals_for_exec.update(self.constants_namespace)
        self.reset()

    @property
    def target_input(self):
        return self.puzzle["test"][0]["input"]

    @property
    def target_output(self):
        return self.puzzle["test"][0]["output"]

    def reset(self):
        self.remaining_step_ids = {step["id"] for step in self.steps}
        self.vars = {"I": self.target_input}
        self.history = []
        self.done = False
        self.step_count = 0
        return self.render_observation(), self.info()

    def info(self) -> dict:
        candidate = self.vars.get("O")
        if candidate is None:
            total = len(self.target_output) * len(self.target_output[0])
            return {
                "score": 0,
                "max_score": total,
                "accuracy": 0.0,
                "done": self.done,
            }

        matched, total = compare_grids(candidate, self.target_output)
        return {
            "score": matched,
            "max_score": total,
            "accuracy": matched / total if total else 0.0,
            "done": self.done,
        }

    def render_train_examples(self) -> str:
        blocks = []
        for i, example in enumerate(self.puzzle["train"], start=1):
            blocks.append(
                f"Example {i} input:\n{grid_to_text(example['input'])}\n\n"
                f"Example {i} output:\n{grid_to_text(example['output'])}"
            )
        return "\n\n".join(blocks)

    def render_helper_dsl(self) -> str:
        lines = []
        for name, doc in self.helper_docs:
            lines.append(f"- {name}: {doc}")
        return "\n".join(lines)

    def render_available_options(self) -> str:
        lines = []
        for step in self.steps:
            if step["id"] in self.remaining_step_ids:
                deps = ", ".join(step["dependencies"])
                lines.append(f"[{step['id']}] {step['source']}    # deps: {deps}")
        return "\n".join(lines)

    def render_history(self) -> str:
        if not self.history:
            return "(none)"
        lines = []
        for entry in self.history[-6:]:
            lines.append(
                f"[{entry['id']}] {entry['source']}\n"
                f"result: {entry['summary']}"
            )
        return "\n\n".join(lines)

    def render_candidate_status(self) -> str:
        candidate = self.vars.get("O")
        if candidate is None:
            return "Current candidate O: (not computed yet)"

        matched, total = compare_grids(candidate, self.target_output)
        return (
            f"Current candidate O:\n{grid_to_text(candidate)}\n\n"
            f"Cell match score: {matched}/{total} ({matched/total:.1%})"
        )

    def render_observation(self, last_result: str | None = None) -> str:
        parts = [
            f"Puzzle ID: {self.puzzle_id}",
            "Training input-output examples:",
            self.render_train_examples(),
            "Target test input:",
            grid_to_text(self.target_input),
            "Helper DSLs used in this puzzle:",
            self.render_helper_dsl(),
            "Executed history:",
            self.render_history(),
            self.render_candidate_status(),
        ]

        if last_result:
            parts.extend(["Last action result:", last_result])

        if self.step_count < self.max_steps and not self.done:
            parts.extend(
                [
                    "Available options:",
                    self.render_available_options(),
                    "Reply with exactly one option number.",
                ]
            )

        return "\n\n".join(parts)

    def execute_option(self, option_id: int) -> str:
        step = next(s for s in self.steps if s["id"] == option_id)
        local_vars = dict(self.vars)
        exec(step["source"], self.globals_for_exec, local_vars)

        result_value = local_vars.get(step["target"])
        self.vars = local_vars
        self.remaining_step_ids.remove(option_id)

        summary = summarize_value(result_value)
        self.history.append(
            {
                "id": option_id,
                "source": step["source"],
                "summary": summary,
            }
        )
        return f"Executed [{option_id}] {step['source']}\nValue summary:\n{summary}"

    def step(self, action_text: str):
        self.step_count += 1
        option_id = parse_option_id(action_text)
        last_result = ""

        if option_id is None:
            last_result = (
                f"Could not parse an option number from: {action_text!r}. "
                "Please reply with a single integer."
            )
        elif option_id not in self.remaining_step_ids:
            last_result = (
                f"Invalid option id: {option_id}. "
                "Choose one of the remaining option numbers."
            )
        else:
            try:
                last_result = self.execute_option(option_id)
            except Exception as exc:
                step = next(s for s in self.steps if s["id"] == option_id)
                last_result = (
                    f"Execution failed for [{option_id}] {step['source']}\n"
                    f"{type(exc).__name__}: {exc}"
                )

        if "O" in self.vars:
            matched, total = compare_grids(self.vars["O"], self.target_output)
            last_result += f"\n\nMatch score after this action: {matched}/{total}"
            if matched == total:
                self.done = True

        if self.step_count >= self.max_steps:
            self.done = True

        obs = self.render_observation(last_result=last_result)
        reward = 1 if ("O" in self.vars and self.vars["O"] == self.target_output) else 0
        return obs, reward, self.done, self.info()



## Agent definition

This mirrors the structure of your in-class notebook:
- `_api_call`
- `get_system_prompt`
- `act`
- retry logic

The only difference is that the model must output **one solver-step option id** instead of a text-adventure command.


In [7]:

class MyARCAgent:
    def __init__(
        self,
        llm: str,
        api_url: str,
        key: str,
        seed: int = 0,
        act_temp: float = 0.0,
        reasoning_effort: int = 1024,
        conversation: bool = True,
    ):
        self.model = llm
        self.api_url = api_url
        self.api_key = key
        self.seed = seed
        self.act_temp = act_temp
        self.reasoning_effort = reasoning_effort
        self.conversation = conversation
        self.history: list[tuple[str, str]] = []

    def reset(self):
        self.history = []

    def _api_call(self, messages, **kwargs):
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }

        payload = {
            "model": self.model,
            "messages": messages,
        }

        if "temperature" in kwargs:
            payload["temperature"] = kwargs["temperature"]
        if "max_tokens" in kwargs:
            payload["max_tokens"] = kwargs["max_tokens"]
            payload["max_completion_tokens"] = kwargs["max_tokens"]
        if "seed" in kwargs:
            payload["seed"] = kwargs["seed"]
        if "reasoning_effort" in kwargs:
            payload["reasoning_effort"] = kwargs["reasoning_effort"]

        response = requests.post(
            self.api_url,
            headers=headers,
            json=payload,
            timeout=300,
        )
        response.raise_for_status()
        return response.json()

    def get_system_prompt(self):
        return (
            "You are an expert ARC-AGI solver.\n"
            "You are playing a step-by-step DSL game.\n"
            "At each turn you will see:\n"
            "- ARC training input-output examples\n"
            "- the target test input grid\n"
            "- the small DSL helper list used in this puzzle\n"
            "- the executed history so far\n"
            "- the remaining executable solver-step options\n\n"
            "Your job is to infer the transformation from the training pairs and "
            "pick the next best solver step.\n"
            "Respect variable dependencies. If an option depends on undefined "
            "variables, avoid it.\n"
            "Reply with EXACTLY ONE option number and nothing else.\n"
            "Reasoning: Low"
        )

    def build_messages(self, current_observation: str):
        messages = [{"role": "system", "content": self.get_system_prompt()}]
        if self.conversation:
            for obs, act in self.history:
                messages.append({"role": "user", "content": obs})
                messages.append({"role": "assistant", "content": act})
        messages.append({"role": "user", "content": current_observation})
        return messages

    def _call_with_retry(
        self,
        messages,
        max_retries: int = 5,
        base_sleep: float = 1.0,
        call_kwargs: dict | None = None,
    ):
        call_kwargs = call_kwargs or {}
        last_error = None
        for attempt in range(max_retries):
            try:
                response = self._api_call(messages, **call_kwargs)
                if "choices" in response:
                    return response
            except Exception as exc:
                last_error = exc

            if attempt < max_retries - 1:
                time.sleep(base_sleep * (2 ** attempt))

        if last_error is not None:
            raise last_error
        raise RuntimeError("API call failed without returning a usable response.")

    def act(self, obs: str, score: int, done: bool, infos: dict):
        messages = self.build_messages(obs)

        call_kwargs = {
            "temperature": self.act_temp,
            "max_tokens": 64,
            "seed": self.seed,
            "reasoning_effort": self.reasoning_effort,
        }

        response = self._call_with_retry(messages, call_kwargs=call_kwargs)

        message = (response.get("choices", [{}])[0].get("message", {}) or {})

        print(f"API response message content:\n{message.get('content')}\n")
        
        content = (message.get("content") or "").strip()

        option_id = parse_option_id(content)
        if option_id is None:
            action = "1"
        else:
            action = str(option_id)

        self.history.append((obs, action))

        usage = response.get("usage", {})
        stats = {
            "response_text": content,
            "action": action,
            "nb_tokens_prompt": usage.get("prompt_tokens", 0),
            "nb_tokens_thinking": usage.get("completion_tokens", 0),
        }
        stats["nb_tokens"] = stats["nb_tokens_prompt"] + stats["nb_tokens_thinking"]
        return action, stats


## Trajectory display helpers

In [8]:

def pretty_print_trajectory(trajectory: dict):
    print("=" * 80)
    print("ARC TRAJECTORY SUMMARY")
    print("=" * 80)
    print(f"Puzzle ID   : {trajectory['puzzle_id']}")
    print(f"Model       : {trajectory['model']}")
    print(f"Seed        : {trajectory['seed']}")
    print(f"Max Steps   : {trajectory['max_steps']}")
    print(f"Best Score  : {trajectory['best_score']} / {trajectory['max_score']}")
    print(f"Solved      : {trajectory['solved']}")
    print()

    print("-" * 80)
    print("SYSTEM PROMPT")
    print("-" * 80)
    print(trajectory["system_prompt"])
    print()

    for entry in trajectory["steps"]:
        print("=" * 80)
        print(
            f"STEP {entry['step']} | action={entry['action']} | "
            f"score={entry['score']}/{entry['max_score']} | done={entry['done']}"
        )
        print("=" * 80)
        print("\n--- Observation ---")
        print(entry["observation"])
        print("\n--- Model raw response ---")
        print(entry["model_response"])
        print()


def run_single_puzzle(
    puzzle_id: str,
    *,
    data_dir: pathlib.Path,
    solvers_path: pathlib.Path,
    dsl_namespace: dict,
    constants_namespace: dict,
    api_url: str,
    api_key: str,
    model: str,
    max_steps: int = 30,
    seed: int = 0,
    temperature: float = 0.0,
    reasoning_effort: int = 1024,
):
    env = ARCStepEnv(
        puzzle_id=puzzle_id,
        data_dir=data_dir,
        solvers_path=solvers_path,
        dsl_namespace=dsl_namespace,
        constants_namespace=constants_namespace,
        max_steps=max_steps,
    )

    agent = MyARCAgent(
        llm=model,
        api_url=api_url,
        key=api_key,
        seed=seed,
        act_temp=temperature,
        reasoning_effort=reasoning_effort,
        conversation=True,
    )

    obs, info = env.reset()

    trajectory = {
        "puzzle_id": puzzle_id,
        "model": model,
        "seed": seed,
        "max_steps": max_steps,
        "system_prompt": agent.get_system_prompt(),
        "steps": [],
        "best_score": 0,
        "max_score": info["max_score"],
        "solved": False,
    }

    best_score = 0
    done = False

    for step_idx in range(1, max_steps + 1):
        action, stats = agent.act(obs, info["score"], done, info)
        print(
            f"Step {step_idx:02d} | option {action} | "
            f"score {info['score']}/{info['max_score']} | "
            f"best {best_score}/{info['max_score']}"
        )

        obs, reward, done, info = env.step(action)
        best_score = max(best_score, info["score"])

        trajectory["steps"].append(
            {
                "step": step_idx,
                "observation": obs,
                "action": action,
                "model_response": stats["response_text"],
                "score": info["score"],
                "max_score": info["max_score"],
                "done": done,
                "tokens_used": stats["nb_tokens"],
                "reasoning_tokens": stats["nb_tokens_thinking"],
            }
        )

        if done:
            break

    trajectory["best_score"] = best_score
    trajectory["solved"] = info["score"] == info["max_score"]
    trajectory["final_candidate"] = env.vars.get("O")
    trajectory["ground_truth"] = env.target_output
    return trajectory


## Pick a puzzle and run the ARC agent

In [14]:

TEST_IDS = PUZZLE_SPLITS["TEST_IDS"]

def puzzle_exists(puzzle_id: str) -> bool:
    try:
        resolve_puzzle_path(DATA_DIR, puzzle_id)
        return True
    except FileNotFoundError:
        return False

AVAILABLE_TEST_IDS = [pid for pid in TEST_IDS if puzzle_exists(pid)]
AVAILABLE_TRAIN_IDS = [pid for pid in PUZZLE_SPLITS["TRAIN_IDS"] if puzzle_exists(pid)]

print(f"Found {len(AVAILABLE_TEST_IDS)} available TEST_IDS")
print(f"Found {len(AVAILABLE_TRAIN_IDS)} available TRAIN_IDS")

if AVAILABLE_TEST_IDS:
    PUZZLE_ID = AVAILABLE_TEST_IDS[0]
elif puzzle_exists("00d62c1b"):
    PUZZLE_ID = "00d62c1b"
else:
    raise FileNotFoundError("No puzzle JSONs found in ./data or workspace root.")

print("Running puzzle:", PUZZLE_ID)


Found 27 available TEST_IDS
Found 104 available TRAIN_IDS
Running puzzle: 0520fde7


In [15]:

if not API_KEY:
    raise ValueError(
        "API key not found. In Colab, set userdata['TritonAPI']; "
        "otherwise export TRITON_API_KEY."
    )

trajectory = run_single_puzzle(
    puzzle_id=PUZZLE_ID,
    data_dir=DATA_DIR,
    solvers_path=SOLVERS_PATH,
    dsl_namespace=DSL_NAMESPACE,
    constants_namespace=CONSTANTS_NAMESPACE,
    api_url=API_URL,
    api_key=API_KEY,
    model=MODEL,
    max_steps=MAX_STEPS,
    seed=SEED,
    temperature=TEMPERATURE,
    reasoning_effort=REASONING_EFFORT,
)

print("\nSolved:", trajectory["solved"])
print("Best score:", trajectory["best_score"], "/", trajectory["max_score"])


HTTPError: 400 Client Error: Bad Request for url: https://tritonai-api.ucsd.edu/v1/chat/completions

In [16]:
pretty_print_trajectory(trajectory)

NameError: name 'trajectory' is not defined


## Optional: evaluate across several puzzles

This is a simple wrapper to run the same agent across multiple available ARC puzzles.
You can point it at `TEST_IDS`, or at a smaller subset first.


In [17]:

def run_many(puzzle_ids, limit=None):
    results = []
    selected = list(puzzle_ids[:limit] if limit is not None else puzzle_ids)

    for puzzle_id in selected:
        if not (DATA_DIR / f"{puzzle_id}.json").exists():
            print(f"Skipping {puzzle_id}: missing data/{puzzle_id}.json")
            continue

        print("\n" + "#" * 80)
        print("Puzzle:", puzzle_id)
        print("#" * 80)

        try:
            traj = run_single_puzzle(
                puzzle_id=puzzle_id,
                data_dir=DATA_DIR,
                solvers_path=SOLVERS_PATH,
                dsl_namespace=DSL_NAMESPACE,
                constants_namespace=CONSTANTS_NAMESPACE,
                api_url=API_URL,
                api_key=API_KEY,
                model=MODEL,
                max_steps=MAX_STEPS,
                seed=SEED,
                temperature=TEMPERATURE,
                reasoning_effort=REASONING_EFFORT,
            )
            results.append(
                {
                    "puzzle_id": puzzle_id,
                    "solved": traj["solved"],
                    "best_score": traj["best_score"],
                    "max_score": traj["max_score"],
                    "steps_used": len(traj["steps"]),
                }
            )
        except Exception as exc:
            results.append(
                {
                    "puzzle_id": puzzle_id,
                    "solved": False,
                    "best_score": 0,
                    "max_score": 0,
                    "steps_used": 0,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

    return results


# Example:
results = run_many(AVAILABLE_TEST_IDS, limit=5)
results



################################################################################
Puzzle: 0520fde7
################################################################################

################################################################################
Puzzle: e3497940
################################################################################

################################################################################
Puzzle: 25ff71a9
################################################################################

################################################################################
Puzzle: aedd82e4
################################################################################

################################################################################
Puzzle: 08ed6ac7
################################################################################


[{'puzzle_id': '0520fde7',
  'solved': False,
  'best_score': 0,
  'max_score': 0,
  'steps_used': 0,
  'error': 'HTTPError: 400 Client Error: Bad Request for url: https://tritonai-api.ucsd.edu/v1/chat/completions'},
 {'puzzle_id': 'e3497940',
  'solved': False,
  'best_score': 0,
  'max_score': 0,
  'steps_used': 0,
  'error': 'HTTPError: 400 Client Error: Bad Request for url: https://tritonai-api.ucsd.edu/v1/chat/completions'},
 {'puzzle_id': '25ff71a9',
  'solved': False,
  'best_score': 0,
  'max_score': 0,
  'steps_used': 0,
  'error': 'HTTPError: 400 Client Error: Bad Request for url: https://tritonai-api.ucsd.edu/v1/chat/completions'},
 {'puzzle_id': 'aedd82e4',
  'solved': False,
  'best_score': 0,
  'max_score': 0,
  'steps_used': 0,
  'error': 'HTTPError: 400 Client Error: Bad Request for url: https://tritonai-api.ucsd.edu/v1/chat/completions'},
 {'puzzle_id': '08ed6ac7',
  'solved': False,
  'best_score': 0,
  'max_score': 0,
  'steps_used': 0,
  'error': 'HTTPError: 400 Cli